In [0]:
# ══════════════════════════════════════
# ANALYSIS — physical_itens_venda_caixa
# Squad 3 — Batch Ecommerce
# ══════════════════════════════════════

# Célula 1 — Carrega utils (já carrega o config internamente)

%run "/Workspace/Repos/luizhpdatasci@gmail.com/merca-data-platform/Squad3/luiz-portacio/utils/00_utils.ipynb"

In [0]:
# Carregar dados do banco

df_itens = consultar_tabela("physical_itens_venda_caixa")
print(f"✅ physical_itens_venda_caixa carregado! Shape: {df_itens.shape}")


In [0]:
# Visão Geral

print("=" * 50)
print("🛒 VISÃO GERAL — physical_itens_venda_caixa")
print("=" * 50)

print(f"\n📐 Shape: {df_itens.shape}")
print(f"   {df_itens.shape[0]} registros | {df_itens.shape[1]} colunas")

print("\n📋 Colunas e tipos:")
print(df_itens.dtypes)

df_itens.describe()


In [0]:
# Análise de Quantidade

print("=" * 50)
print("📊 ANÁLISE DE QUANTIDADE")
print("=" * 50)

print(f"\n   Total vendido  : {df_itens['quantidade'].sum():.2f}")
print(f"   Média por item : {df_itens['quantidade'].mean():.2f}")
print(f"   Máximo         : {df_itens['quantidade'].max():.2f}")
print(f"   Mínimo         : {df_itens['quantidade'].min():.2f}")

print("\n📊 Distribuição de quantidade:")
print(df_itens['quantidade'].value_counts().head(10).to_string())


In [0]:
# Análise de Valores

print("=" * 50)
print("💰 ANÁLISE DE VALORES")
print("=" * 50)

print(f"\n   Receita total          : R$ {df_itens['valor_total_item'].sum():,.2f}")
print(f"   Ticket médio por item  : R$ {df_itens['valor_total_item'].mean():,.2f}")
print(f"   Valor máximo           : R$ {df_itens['valor_total_item'].max():,.2f}")
print(f"   Valor mínimo           : R$ {df_itens['valor_total_item'].min():,.2f}")

print(f"\n   Preço unitário médio   : R$ {df_itens['preco_unitario_registro'].mean():,.2f}")
print(f"   Preço unitário máximo  : R$ {df_itens['preco_unitario_registro'].max():,.2f}")
print(f"   Preço unitário mínimo  : R$ {df_itens['preco_unitario_registro'].min():,.2f}")


In [0]:
# Top Produtos

print("=" * 50)
print("🏆 TOP 10 PRODUTOS POR RECEITA")
print("=" * 50)

top_receita = df_itens.groupby('codigo_barras_produto').agg(
    receita_total  = ('valor_total_item', 'sum'),
    qtd_vendida    = ('quantidade', 'sum'),
    preco_medio    = ('preco_unitario_registro', 'mean')
).reset_index() \
 .sort_values('receita_total', ascending=False) \
 .head(10)

top_receita['receita_total'] = top_receita['receita_total'].map('R$ {:,.2f}'.format)
top_receita['preco_medio']   = top_receita['preco_medio'].map('R$ {:,.2f}'.format)
print(top_receita.to_string(index=False))


In [0]:
# Top Produtos por Quantidade

print("=" * 50)
print("🏆 TOP 10 PRODUTOS POR QUANTIDADE VENDIDA")
print("=" * 50)

top_qtd = df_itens.groupby('codigo_barras_produto').agg(
    qtd_vendida   = ('quantidade', 'sum'),
    receita_total = ('valor_total_item', 'sum')
).reset_index() \
 .sort_values('qtd_vendida', ascending=False) \
 .head(10)

top_qtd['receita_total'] = top_qtd['receita_total'].map('R$ {:,.2f}'.format)
print(top_qtd.to_string(index=False))

In [0]:
# Análise por Transação

print("=" * 50)
print("🧾 ANÁLISE POR TRANSAÇÃO")
print("=" * 50)

por_transacao = df_itens.groupby('id_transacao').agg(
    total_itens    = ('id_item_venda', 'count'),
    valor_total    = ('valor_total_item', 'sum'),
    qtd_total      = ('quantidade', 'sum')
).reset_index()

print(f"\n   Total de transações         : {len(por_transacao):,}")
print(f"   Média de itens por transação: {por_transacao['total_itens'].mean():.2f}")
print(f"   Ticket médio por transação  : R$ {por_transacao['valor_total'].mean():,.2f}")
print(f"   Maior transação             : R$ {por_transacao['valor_total'].max():,.2f}")
print(f"   Menor transação             : R$ {por_transacao['valor_total'].min():,.2f}")

In [0]:
# Conclusões

print("=" * 50)
print("📋 CONCLUSÕES — physical_itens_venda_caixa")
print("=" * 50)

print(f"""
   Total de registros          : {len(df_itens):,}
   Total de transações         : {por_transacao.shape[0]:,}
   Total de produtos únicos    : {df_itens['codigo_barras_produto'].nunique():,}
   Receita total               : R$ {df_itens['valor_total_item'].sum():,.2f}
   Ticket médio por transação  : R$ {por_transacao['valor_total'].mean():,.2f}
   Produto mais vendido (qtd)  : {df_itens.groupby('codigo_barras_produto')['quantidade'].sum().idxmax()}
   Produto maior receita       : {df_itens.groupby('codigo_barras_produto')['valor_total_item'].sum().idxmax()}
""")